# 📗 임베딩 파인튜닝 심화: MNR 계산과 Hard Negative 마이닝

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">Hard Negative</strong>(헷갈리는 오답 문서)는 질의와 표현이나 주제가 비슷해서 모델이 높은 점수를 주지만, 질의가 묻는 내용에는 맞지 않는 문서입니다. 이런 문서를 찾아 학습 데이터에 넣는 작업을 <strong style="font-size:1.13em;color:#1f3a5f;">Hard Negative 마이닝</strong>(mining)이라 합니다. 실제로도 답이 되는 문서를 오답으로 넣지 않도록 내용 검토가 필요합니다.</p></div>

<img src="images/30_hard_negative.png" width="1050" alt="쌍 학습 뒤 남은 헷갈리는 오답을 검토해 정답과 구분하도록 학습하는 목적.">

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01의 쌍 학습 뒤에도 표현이 비슷한 오답을 높은 순위로 찾을 수 있습니다. 이 교안에서는 그런 오답을 수집·검토해 정답과 더 잘 구분하도록 새 모델을 훈련합니다. 같은 MNR 손실의 계산을 이해하고, 도메인·일반 분야 검색을 함께 비교해 학습 효과와 퇴행을 판단합니다.</p></div>

<img src="images/31_lesson02_goals.png" width="1050" alt="학습 목표: 1 MNR 계산, 2 어려운 오답 수집, 3 거짓 오답 검토, 4 심화 학습과 평가.">

**학습 목표** (그림의 번호와 같습니다)

1. **작은 배치의 유사도·확률로 MNR 손실을 계산하고 라이브러리 결과와 비교합니다.**
2. **학습 질의에서 정답 조문을 제외하고 헷갈리는 오답 후보를 수집합니다.**
3. **점수 규칙과 원문 검토를 이용해 거짓 오답이 의심되는 학습 행을 제외합니다.**
4. **Triplet 데이터로 학습한 모델을 기존 모델과 일반화·퇴행 관점에서 비교합니다.**

## 1. 교안 01의 모델과 데이터 준비

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01에서 검증 Recall@5가 가장 높아 복원·저장한 <code style="font-size:0.95em;color:#345473;">output/labor_tuned</code>를 불러옵니다. 이 교안은 새 커널에서 시작하며, 교안 01의 쌍 학습을 다시 실행하지 않습니다. 함께 따라하기까지 진행하려면 <code style="font-size:0.95em;color:#345473;">output/med_tuned</code>도 필요합니다.</p></div>

### 1-1. 경로와 계산 장치

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">경로·장치 선택은 교안 01과 같은 코드를 사용합니다.</p></div>

In [ ]:
from pathlib import Path

import pandas as pd
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 조문이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 이 단원의 임베딩 모델과 e5 계열 프리픽스(질의는 "query: ", 문서는 "passage: ")
MODEL_ID = "intfloat/multilingual-e5-small"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

### 1-2. 제공 함수와 기존 데이터

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01의 <code style="font-size:0.95em;color:#345473;">load_jsonl</code>, <code style="font-size:0.95em;color:#345473;">encode_texts</code>, <code style="font-size:0.95em;color:#345473;">gold_ranks</code>, <code style="font-size:0.95em;color:#345473;">retrieval_scores</code>를 그대로 씁니다. 이 셀은 실행만 하고 다음으로 넘어갑니다.</p></div>

In [ ]:
import json


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def encode_texts(model, texts, prefix):
    """텍스트 목록에 프리픽스를 붙여 정규화된 임베딩(numpy 배열)으로 만듭니다."""
    return model.encode(
        [prefix + text for text in texts],   # 프리픽스를 붙인 문장 목록
        normalize_embeddings=True,           # 길이 1로 정규화해 벡터끼리의 내적이 곧 코사인 유사도가 되게 합니다
        show_progress_bar=False,             # 진행 막대를 끕니다(평가 때마다 여러 번 불러 출력이 길어지지 않게)
    )


import numpy as np


def gold_ranks(model, corpus, evalset, query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX):
    """평가 질의마다 정답 문서가 유사도 순으로 몇 위인지(1위부터) 담은 배열을 돌려줍니다."""
    doc_ids = [doc["doc_id"] for doc in corpus]
    doc_emb = encode_texts(model, [doc["text"] for doc in corpus], passage_prefix)
    # 학습·평가 파일은 같은 세 열을 쓰며, 검색 평가는 질의와 정답 ID를 사용합니다.
    query_emb = encode_texts(model, [row["anchor"] for row in evalset], query_prefix)

    # 정규화한 벡터끼리의 내적이 코사인 유사도입니다. sims는 (질의 수, 문서 수) 행렬입니다
    sims = query_emb @ doc_emb.T

    # 질의마다 유사도가 높은 순으로 문서 id를 늘어놓고, 정답 문서가 몇 번째인지 셉니다
    ranks = []
    for i, row in enumerate(evalset):
        ranked_ids = [doc_ids[j] for j in np.argsort(-sims[i])]
        ranks.append(ranked_ids.index(row["doc_id"]) + 1)
    return np.array(ranks)


def retrieval_scores(ranks):
    """정답 순위 배열로 Recall@1, Recall@5, MRR을 계산합니다."""
    return {
        "Recall@1": round(float(np.mean(ranks <= 1)), 3),  # 정답 문서가 1위인 질의의 비율
        "Recall@5": round(float(np.mean(ranks <= 5)), 3),  # 정답 문서가 5위 안에 든 질의의 비율
        "MRR": round(float(np.mean(1 / ranks)), 3),        # 평균(1 / 정답 순위)
    }

### 1-3. 모델과 학습 쌍

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">마이닝은 <strong style="font-size:1.13em;color:#1f3a5f;">학습 질의</strong>로만 진행합니다. 검증 질의는 마이닝·가중치 학습에 넣지 않고, 매 에폭 검색 평가와 최적 에폭 선택에 사용합니다.</p></div>

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer

# 교안 01과 같은 노무 데이터입니다. 평가 질의는 마이닝에 넣지 않습니다
labor_corpus = load_jsonl(data_dir / "본실습" / "labor_corpus.jsonl")
labor_train = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
labor_eval = load_jsonl(data_dir / "본실습" / "labor_eval.jsonl")

# 베이스 모델은 비교용이고, 저장한 쌍 학습 모델은 마이닝에 사용합니다
base_model = SentenceTransformer(MODEL_ID, device=device)
tuned_model = SentenceTransformer(str(output_dir / "labor_tuned"), device=device)

# 데이터 열 순서와 프리픽스는 교안 01과 같습니다
pair_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in labor_train],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in labor_train],
})
display(pair_ds)

## 2. MNR 손실을 배치 하나로 계산하기

### 2-1. 유사도 행렬과 정답 위치

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01의 MNR 손실이 어떤 값을 줄이는지 네 쌍으로 계산합니다. 행은 질의, 열은 각 쌍의 정답 조문이므로 <strong style="font-size:1.13em;color:#1f3a5f;">대각선이 자기 정답</strong>입니다. 같은 행의 나머지 세 칸은 배치 내 오답 후보입니다.</p></div>

<img src="images/34_mnr_loss_meaning.png" width="1050" alt="정답 확률이 낮으면 손실이 크고, 높으면 손실이 작아지는 MNR 손실의 의미.">

In [ ]:
# 학습 쌍에서 166행 간격으로 4쌍을 고릅니다. 정답 조문이 서로 다른 4쌍입니다
batch_rows = labor_train[::166]
batch_table = pd.DataFrame(batch_rows)[["anchor", "doc_id"]]
display(batch_table)

In [ ]:
# 질의와 정답 조문을 베이스 모델로 인코딩합니다
batch_query_emb = base_model.encode(
    [QUERY_PREFIX + row["anchor"] for row in batch_rows],
    normalize_embeddings=True,
    convert_to_tensor=True,   # 손실 함수에 넣으려고 numpy 대신 torch 텐서로 받습니다
)
batch_doc_emb = base_model.encode(
    [PASSAGE_PREFIX + row["positive"] for row in batch_rows],
    normalize_embeddings=True,
    convert_to_tensor=True,
)

# 4 x 4 코사인 유사도 행렬: 행은 질의, 열은 정답 조문. 대각선 칸이 자기 정답 조문입니다
batch_sims = batch_query_emb @ batch_doc_emb.T
row_names = ["질의1", "질의2", "질의3", "질의4"]
col_names = ["정답 조문1", "정답 조문2", "정답 조문3", "정답 조문4"]
sims_table = pd.DataFrame(batch_sims.cpu().numpy(), index=row_names, columns=col_names)
display(sims_table.round(3))

### 2-2. Softmax로 점수를 확률로 바꾸기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">Softmax</strong>는 같은 질의가 비교하는 문서들의 점수를 합이 1인 확률로 바꿉니다. 코사인 유사도에 <code style="font-size:0.95em;color:#345473;">scale=20</code>을 곱한 뒤 행마다 적용하며, 대각선이 자기 정답의 확률입니다. 이 20은 Sentence Transformers 손실 함수의 기본값이며, 보편적으로 고정된 상수는 아닙니다.</p></div>

$\displaystyle s_{ij}=20\cos(q_i,d_j),\qquad p_i=\frac{\exp(s_{ii})}{\sum_{j=1}^{B}\exp(s_{ij})}$

$B$는 배치의 쌍 수이며 여기서는 4입니다. $q_i$는 질의 $i$의 벡터, $d_j$는 후보 문서 $j$의 벡터이고 $d_i$가 자기 정답입니다. $p_i$는 이 후보들 중 정답에 준 확률이며 검색 정확도는 아닙니다.

In [ ]:
# 유사도에 scale 20 을 곱해 점수로 만들고, 행마다 softmax 로 확률을 만듭니다
batch_scores = batch_sims * 20
batch_probs = torch.softmax(batch_scores, dim=1)   # dim=1: 행(질의)마다 확률의 합이 1

# 대각선 칸이 질의가 자기 정답 조문을 고를 확률입니다
probs_table = pd.DataFrame(batch_probs.cpu().numpy(), index=row_names, columns=col_names)
display(probs_table.round(3))

### 2-3. 크로스 엔트로피로 정답 확률에 벌점 주기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">크로스 엔트로피(cross entropy)</strong> 는 정답 문서에 준 확률의 <code style="font-size:0.95em;color:#345473;">−log</code>로 손실을 계산합니다. MNR은 각 질의의 크로스 엔트로피를 배치 전체에서 평균냅니다.</p></div>

$\displaystyle \ell_i=-\log p_i,\qquad L_{\mathrm{MNR}}=\frac{1}{B}\sum_{i=1}^{B}\ell_i$

$\ell_i$는 질의 $i$ 하나의 손실, $L_{\mathrm{MNR}}$은 배치의 평균 손실입니다. 정답 확률 0.8의 손실은 약 0.223, 0.2의 손실은 약 1.609로, 정답을 놓칠수록 벌점이 커집니다.

즉 **유사도 점수 → Softmax 확률 → 정답 확률의 −log → 배치 평균**입니다. 아래 `cross_entropy`는 확률이 아닌 `batch_scores`를 받아 내부에서 같은 계산을 안정적으로 처리하므로 Softmax를 두 번 적용하지 않습니다.

In [7]:
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

# 행 i 의 정답 위치는 i 번째 열(대각선)입니다
labels = torch.tensor([0, 1, 2, 3], device=device)

# 수식 그대로: 대각선 정답 확률의 -log를 질의 전체에서 평균냅니다
manual_loss = -torch.log(torch.diagonal(batch_probs)).mean()

# cross_entropy는 확률 대신 점수를 받아 log_softmax와 같은 손실 계산을 내부에서 처리합니다
cross_entropy_loss = torch.nn.functional.cross_entropy(batch_scores, labels)

# 같은 임베딩을 MNR 손실 함수에 넣습니다
mnr_loss = MultipleNegativesRankingLoss(base_model)
library_loss = mnr_loss.compute_loss_from_embeddings(
    [batch_query_emb, batch_doc_emb],   # 질의 임베딩, 정답 조문 임베딩 순서
    None,                               # 라벨: MNR 손실은 대각선이 정답이라 따로 받지 않습니다
)

loss_compare = pd.DataFrame(
    {"손실": [manual_loss.item(), cross_entropy_loss.item(), library_loss.item()]},
    index=["직접 수식: -log(정답 확률)의 평균", "크로스 엔트로피: 점수와 정답 위치", f"MNR 손실(scale {mnr_loss.scale})"],
)
display(loss_compare.round(6))

,손실
직접 수식: -log(정답 확률)의 평균,0.832651
크로스 엔트로피: 점수와 정답 위치,0.832651
MNR 손실(scale 20.0),0.832651


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>세 값이 같으면 직접 수식, PyTorch 크로스 엔트로피, MNR이 같은 손실을 계산한 것입니다. 이 셀은 값을 비교하며, 파라미터 학습은 하지 않습니다.</li>
</ul></div>

## 3. 헷갈리는 오답 후보 직접 고르기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">질의와 주제가 다른 쉬운 오답보다, 표현은 비슷하지만 묻는 조건이 다른 문서를 구분하는 연습이 필요합니다. 다만 <strong style="font-size:1.13em;color:#1f3a5f;">유사도가 높고 정답 ID가 다르다는 사실만으로 실제 오답이 확정되지는 않습니다.</strong> 직접 고른 문서는 먼저 오답 후보로 봅니다.</p></div>

### 3-1. 학습 질의와 조문의 유사도

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">기존 노무 학습 질의 하나로 조문 전체를 검색합니다. 평가 질의를 학습 데이터로 바꾸지 않습니다.</p></div>

In [ ]:
# 조문 ID와 벡터의 행 순서를 맞춰 둡니다
# 이 질의는 평가셋이 아니라 학습 데이터에서 고릅니다
query_idx = 240
doc_ids = [doc["doc_id"] for doc in labor_corpus]
base_doc_emb = encode_texts(base_model, [doc["text"] for doc in labor_corpus], PASSAGE_PREFIX)
query_sims = encode_texts(base_model, [labor_train[query_idx]["anchor"]], QUERY_PREFIX) @ base_doc_emb.T

# 유사도가 높은 순서로 보되, 정답 조문 자체는 오답 후보에서 제외합니다
for j in np.argsort(-query_sims[0]):
    if doc_ids[j] != labor_train[query_idx]["doc_id"]:
        hard_neg_text = labor_corpus[j]["text"]
        hard_neg_id = doc_ids[j]
        break

print("질의:", labor_train[query_idx]["anchor"])
print("정답 조문:", labor_train[query_idx]["doc_id"], "|", labor_train[query_idx]["positive"])
print("오답 후보:", hard_neg_id, "|", hard_neg_text)

### 3-2. 정답과 오답 후보의 원문 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">두 조문이 다루는 제도와 조건을 대조합니다. 육아휴직을 이유로 한 불리한 처우를 묻는 질의에 육아기 근로시간 단축을 다루는 조문이 나왔다면, 표현은 비슷하지만 적용 제도가 다른 경우입니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 퀴즈</strong>: 정답 조문을 제외하지 않고 유사도 1위 조문을 오답 칸에 넣으면 어떤 문제가 생길 수 있습니까?</p></div>

<details><summary>정답 보기</summary>

정답이 이미 1위인 질의에서는 같은 조문이 정답 칸과 오답 칸에 함께 들어갑니다. 모델에 같은 문서를 가깝게 하면서 멀게 하라는 충돌 신호를 주므로, 정답을 후보에서 제외한 뒤 다른 문서도 정말 오답인지 검토합니다.

</details>

## 4. 마이닝한 후보를 검토해 학습 데이터 만들기

### 4-1. 점수 간격으로 후보 선별

`mine_hard_negatives`는 이 모델의 **코사인 유사도**로 질의와 가까운 문서를 찾습니다. 점수 함수 인자를 지정하지 않았으므로 이 실습 모델의 기본 유사도인 cosine을 사용합니다. 정답 점수와 비교해 조건을 통과한 후보 하나를 질의마다 붙입니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">점수 간격 규칙</strong></p>
<ul>
<li><code style="font-size:0.95em;color:#345473;">relative_margin=0.05</code>는 후보 점수가 정답 점수보다 <strong style="font-size:1.13em;color:#1f3a5f;">정답 점수 절댓값의 5% 이상 낮아야</strong> 한다는 뜻입니다.</li>
<li>정답 점수가 양수라면 기준선은 정답 점수의 95%입니다. 예를 들어 정답 점수가 0.8이면 기준선은 0.76입니다. 이 숫자는 규칙을 설명하는 예시입니다.</li>
<li>정답 점수에 지나치게 가까운 후보는 거짓 오답일 수 있어 제외합니다. 기준을 통과한 문서는 Hard Negative 후보이며, 실제 오답인지는 다음 절에서 원문으로 확인합니다.</li>
</ul></div>

[공식 마이닝 문서](https://www.sbert.net/docs/package_reference/util/hard_negatives.html)의 점수 간격 규칙을 사용합니다. 질의와 문서에는 이미 프리픽스가 붙어 있으므로 프리픽스 인자를 더 주지 않습니다.

<img src="images/32_mining_review.png" width="1050" alt="후보 검색, 정답 제외, 점수 선별, 내용 검토를 거쳐 학습 행을 만드는 과정.">

In [ ]:
from sentence_transformers.util import mine_hard_negatives

# 문서 후보에도 학습 쌍과 같은 passage 프리픽스를 한 번만 붙입니다
corpus_texts = [PASSAGE_PREFIX + doc["text"] for doc in labor_corpus]
triplet_ds = mine_hard_negatives(
    pair_ds,
    tuned_model,                 # 질의/정답 쌍으로 학습한 모델이 후보 점수를 매깁니다
    corpus=corpus_texts,
    relative_margin=0.05,         # 후보 점수 <= 정답 점수 - abs(정답 점수) * 0.05
    num_negatives=1,              # 질의마다 검토할 후보를 하나씩 붙입니다
    sampling_strategy="top",     # 조건을 통과한 후보 중 점수가 가장 높은 것을 고릅니다
    batch_size=64,                # 한 번에 인코딩할 문장 수이며 학습 배치와 다릅니다
    verbose=False,
)
# 마지막 선택 실험에서 마이닝 모델별 남은 행 수를 비교할 때 씁니다
tuned_mined_count = len(triplet_ds)
print("오답 후보를 받은 행:", len(triplet_ds), "/", len(pair_ds))

### 4-2. 학습 행의 원문 검토

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">마이닝은 가까운 상위 후보 범위 안에서 찾으므로, 그 범위에서 조건을 통과한 후보가 없으면 해당 질의는 결과에서 빠질 수 있습니다. 행 수뿐 아니라 질의·정답·오답 후보의 본문을 함께 읽고, 어떤 차이를 학습시키는지 확인합니다.</p></div>

In [ ]:
# 프리픽스를 붙인 문서 본문으로 조문의 ID와 원문을 찾습니다
passage_to_doc = {PASSAGE_PREFIX + doc["text"]: doc for doc in labor_corpus}
triplet_rows = []
for row in triplet_ds:
    pos_doc = passage_to_doc[row["positive"]]
    neg_doc = passage_to_doc[row["negative"]]
    triplet_rows.append({
        "질의": row["anchor"][len(QUERY_PREFIX):],
        "정답 조문": pos_doc["doc_id"],
        "정답 본문": pos_doc["text"],
        "오답 후보 조문": neg_doc["doc_id"],
        "오답 후보 본문": neg_doc["text"],
    })
# 본문을 자르지 않고 세 행만 함께 읽습니다
triplet_table = pd.DataFrame(triplet_rows)
display(triplet_table.head(3))

### 4-3. 거짓 오답 의심 행 제외

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">거짓 오답</strong>(false negative)은 실제로는 질의에 맞는 문서를 오답으로 표시한 경우입니다. 데이터에 정답 ID가 하나만 있어도, 관련되는 다른 조문이 있을 수 있습니다.</p></div>

점수만으로 해결할 수 없어 다음 두 기준에 걸린 행은 오답 학습에서 보수적으로 제외합니다.
- **검토 목록:** 함께 관련될 가능성이 있다고 검토한 조문 짝입니다. 이 예제에서는 퇴직급여법 제9조 제1항과 근로기준법 제36조를 목록에 둡니다.
- **다른 법 조문의 인용:** 정답과 후보 중 한쪽이 다른 쪽의 조 전체 또는 같은 항을 인용하면 보류합니다. 인용이 있다고 모든 질의에서 정답이 같은 것은 아니며, 이 규칙은 법률 정답 판정기가 아닙니다.

In [ ]:
import re

# 어느 쪽이 정답 칸에 와도 검토 목록에서 찾도록 두 순서를 기록합니다
same_answer_pairs = [
    ("퇴직급여법 제9조 제1항", "근로기준법 제36조"),
    ("근로기준법 제36조", "퇴직급여법 제9조 제1항"),
]


def cites(citing_doc, cited_doc):
    '''citing_doc(인용하는 문서)이 cited_doc(인용 대상)의 조 전체나 같은 항을 인용하는지 검사합니다.'''
    if citing_doc["law"] == cited_doc["law"]:
        return False

    # 예: '근로기준법 제2조 제1항'에서 조 번호와 항 번호를 나눕니다
    parts = cited_doc["doc_id"].split(" ")
    article = parts[1]
    clause = None
    if len(parts) == 3:
        clause = parts[2]

    # 인용문에서 법률명 뒤의 조 번호(group 1)와 항 번호(group 2)를 찾습니다
    for match in re.finditer(cited_doc["law"] + r"」?\s*(제\d+조(?:의\d+)?)(?:\s*(제\d+항))?", citing_doc["text"]):
        # 인용된 조 번호가 cited_doc의 조 번호와 다르면 다음 인용을 확인합니다
        if match.group(1) != article:
            continue
        # group(2)가 None이면 조 전체 인용, cited_doc의 항 번호와 같으면 같은 항 인용입니다
        if match.group(2) is None or match.group(2) == clause:
            return True
    return False

In [ ]:
# 검토 목록이나 인용 규칙에 걸린 행과 학습에 남길 행을 나눕니다
kept_rows = []
removed_rows = []
for row in triplet_ds:
    pos_doc = passage_to_doc[row["positive"]]
    neg_doc = passage_to_doc[row["negative"]]
    in_list = (pos_doc["doc_id"], neg_doc["doc_id"]) in same_answer_pairs

    # 검토 목록에 있거나 두 조문이 서로를 인용하면 거짓 오답 의심 행으로 제외합니다
    if in_list or cites(neg_doc, pos_doc) or cites(pos_doc, neg_doc):
        removed_rows.append({"질의": row["anchor"][len(QUERY_PREFIX):], "정답 조문": pos_doc["doc_id"], "오답 후보 조문": neg_doc["doc_id"]})
    else:
        kept_rows.append(row)

# 이후 학습에는 검토에서 남은 행만 사용합니다
triplet_ds = Dataset.from_list(kept_rows)
print("제외한 행:", len(removed_rows), "| 학습에 쓸 행:", len(triplet_ds))
removed_table = pd.DataFrame(removed_rows)
display(removed_table)

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>제외한 행은 거짓 오답일 가능성이 있어 학습에서 뺀 것입니다. 질문·정답·후보 원문을 대조해 후보가 실제로 다른 답인지 확인합니다.</li>
<li><code style="font-size:0.95em;color:#345473;">NO_DUPLICATES</code>는 배치에서 문자열이 완전히 같은 질의·문장의 중복을 제한합니다. 표현이 다른 동의 문장이나 같은 답을 가리키는 문서는 찾아내지 않습니다.</li>
</ul></div>

## 5. Triplet 데이터로 MNR 학습

### 5-1. 데이터 세 칸과 배치 전체의 후보

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">Triplet은 (질의, 정답 문서, 오답 문서) 세 칸의 데이터 형식</strong>입니다. 이 교안은 손실을 <code style="font-size:0.95em;color:#345473;">TripletLoss</code>로 바꾸지 않고, 교안 01과 같은 MNR 손실을 사용합니다.</p></div>

MNR은 배치의 정답 문서뿐 아니라 명시적 오답 문서도 후보에 더합니다. 2행 배치라면 질의 하나에 자기 정답 1개, 다른 행의 정답 1개, 두 행의 오답 2개가 보입니다.

<img src="images/33_triplet_candidates.png" width="1050" alt="2행 triplet에서 한 질의가 비교하는 정답 1개, 배치 내 오답 1개, 명시적 오답 2개.">

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">실제 배치가 32행이면 쌍 학습은 정답 포함 32개, Triplet 학습은 정답 포함 64개를 비교합니다. <code style="font-size:0.95em;color:#345473;">NO_DUPLICATES</code>가 같은 문장의 중복을 막으므로 마지막 배치나 덜 찬 배치에서는 후보 수가 더 적습니다.</p></div>

Triplet에서는 질문·정답·오답 세 열의 문자열을 모두 검사해, 다른 행과 같은 문자열이 있으면 다른 배치로 나눕니다. 의미가 비슷한 문장까지 검사하는 기능은 아닙니다.

후보 수와 구성이 달라지므로 쌍 학습과 Triplet 학습의 평균 손실 크기를 직접 비교해 검색 품질의 우열을 정하지 않습니다.

### 5-2. 모델과 손실

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">쌍 학습 모델에서 이어 학습하지 않고 <strong style="font-size:1.13em;color:#1f3a5f;">같은 베이스 모델에서 새로 시작</strong>합니다. 시작점과 주요 학습 설정은 맞추되, 검토 후 데이터 수와 배치 구성·갱신 횟수까지 같은 실험은 아닙니다.</p></div>

[MNR 공식 문서](https://www.sbert.net/docs/package_reference/sentence_transformer/losses.html#multiplenegativesrankingloss)에서 쌍과 Triplet 형식을 같은 손실이 받는 것을 확인할 수 있습니다.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sentence_transformers.util import cos_sim
from transformers import EarlyStoppingCallback

# 마이닝에 쓴 tuned_model과 별개로, 베이스 모델에서 새 학습을 시작합니다
hard_model = SentenceTransformer(MODEL_ID, device=device)
loss = MultipleNegativesRankingLoss(hard_model)

# 검증 손실은 학습에 넣지 않은 정답 쌍으로 계산합니다.
eval_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in labor_eval],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in labor_eval],
})
# 검증 질의마다 전체 문서에서 정답을 검색해 Recall@5를 계산합니다.
retrieval_evaluator = InformationRetrievalEvaluator(
    # 질의 ID는 정답 매핑과 연결할 문자열 키이고, 값은 프리픽스가 붙은 질문입니다.
    queries={str(i): QUERY_PREFIX + row["anchor"] for i, row in enumerate(labor_eval)},
    # 문서 ID -> 프리픽스가 붙은 검색 문서입니다.
    corpus={doc["doc_id"]: PASSAGE_PREFIX + doc["text"] for doc in labor_corpus},
    # 질의 ID -> 정답 조문 ID 집합입니다.
    relevant_docs={str(i): {row["doc_id"]} for i, row in enumerate(labor_eval)},
    name="labor",  # 지표 이름에 포함되어 eval_labor_cosine_recall@5를 만듭니다.
    score_functions={"cosine": cos_sim},  # 문서 점수는 코사인 유사도로 비교합니다.
    precision_recall_at_k=[1, 5],  # Recall@1과 Recall@5를 계산합니다.
    batch_size=64,  # 평가 문장을 64개씩 인코딩합니다.
    write_csv=False,
)

### 5-3. 학습 설정과 실행

쌍 배치의 수식은 문서 후보가 $B$개이며, 명시적 오답을 한 개씩 더한 Triplet 배치는 후보가 $2B$개가 됩니다. 후보 수가 늘어도 **Softmax → 정답 확률의 크로스 엔트로피**라는 MNR 계산 원리는 같습니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01과 같은 최대 5에폭·배치 32·학습률 <code style="font-size:0.95em;color:#345473;">5e-5</code>로 학습합니다. 매 에폭 검증·저장하고 검증 Recall@5 최적 모델을 복원하며, 2회 연속 미개선이면 종료합니다. 검증 손실도 기록하지만 선택 기준은 검색 Recall@5입니다. 같은 Recall@5는 개선으로 세지 않으며, 동점이면 먼저 선택한 체크포인트를 유지합니다.</p></div>

이 검증셋은 모델 선택용이며 독립 테스트셋이 아닙니다. 일반 83질의는 선택 지표에 사용하지 않습니다.

In [ ]:
# 주요 설정은 쌍 학습과 맞추지만, 남은 행 수·배치 구성·실제 갱신 횟수는 다릅니다.
args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir / "labor_hard_run"),
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_steps=0.1,                           # 전체 갱신의 처음 10% 동안 학습률을 올립니다
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # 질문·정답·오답 문자열을 모두 검사해 배치 내 중복을 피합니다.
    eval_strategy="epoch",                    # 매 에폭 검증 손실과 검색 점수를 계산합니다.
    save_strategy="epoch",
    load_best_model_at_end=True,               # 종료 후 검증 Recall@5 최적 모델을 복원합니다.
    # evaluator의 name=labor, cosine 점수, Recall@5로 만들어진 지표 키입니다.
    metric_for_best_model="eval_labor_cosine_recall@5",
    greater_is_better=True,                    # Recall@5는 값이 클수록 좋은 모델입니다.
    save_total_limit=2,
    logging_steps=7,
    report_to="none",
    seed=42,
    use_cpu=(device.type == "cpu"),             # GPU(cuda)이면 False, CPU이면 True입니다.
)

# 바뀐 핵심은 마이닝·검토한 triplet_ds를 학습 데이터로 넘기는 부분입니다
trainer = SentenceTransformerTrainer(
    model=hard_model,
    args=args,
    train_dataset=triplet_ds,
    eval_dataset=eval_ds,
    evaluator=retrieval_evaluator,
    loss=loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# train_output.global_step과 training_loss로 실제 갱신 횟수와 평균 학습 손실을 봅니다
# 토크나이저 PAD/BOS/EOS 안내가 나와도 학습 진행 여부와 오류 메시지를 구분합니다
train_output = trainer.train()
display(train_output)

# 매 에폭의 검증 손실·검색 점수와 복원한 체크포인트를 읽습니다.
epoch_history = pd.DataFrame([
    {"에폭": log["epoch"], "검증 MNR 손실": round(log["eval_loss"], 4),
     "검증 Recall@5": round(log["eval_labor_cosine_recall@5"], 3)}
    for log in trainer.state.log_history if "eval_labor_cosine_recall@5" in log
])
display(epoch_history)

# 복원된 최적 체크포인트와 선택 점수를 확인합니다.
best_checkpoint_table = pd.DataFrame([{
    "선택 기준": "검증 Recall@5 최대", "최적 Recall@5": round(trainer.state.best_metric, 3),
    "복원 체크포인트": trainer.state.best_model_checkpoint,
}])
display(best_checkpoint_table)

## 6. 전체 점수와 일반화·퇴행 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">아래 노무 점수는 최적 에폭 선택에도 사용한 검증셋의 결과이며 독립 테스트 성능이 아닙니다.</p></div>

### 6-1. 평가 질의를 조문 경험에 따라 나누기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">전체 점수만 보면 기존 조문을 더 잘 찾는 변화와 처음 보는 조문을 덜 찾는 변화가 섞일 수 있습니다. <strong style="font-size:1.13em;color:#1f3a5f;">처음 보는 조문 질의</strong>는 정답 조문이 원래 쌍 학습의 정답에 없던 질의이며, 검색 대상 조문 목록에서 뺐다는 뜻은 아닙니다. 학습에 나온 조문 질의도 평가용으로 남겨 둔 별도 질문입니다.</p></div>

마이닝은 전체 조문을 후보로 사용하므로, Triplet 모델은 이 조문을 오답 후보로 접했을 수 있습니다. 따라서 이 묶음을 Triplet 모델까지 문서가 완전히 미노출된 평가로 해석하지 않습니다.

<img src="images/05_eval_split.png" width="1050" alt="쌍 학습 정답에 있던 조문·없던 조문과 다른 분야의 검색을 나눠 평가하는 기준.">

In [ ]:
# 학습 쌍에 나타난 정답 조문 ID로 평가 질의의 묶음을 만듭니다
train_doc_ids = {row["doc_id"] for row in labor_train}
is_unseen = np.array([row["doc_id"] not in train_doc_ids for row in labor_eval])

base_ranks = gold_ranks(base_model, labor_corpus, labor_eval)
tuned_ranks = gold_ranks(tuned_model, labor_corpus, labor_eval)
hard_ranks = gold_ranks(hard_model, labor_corpus, labor_eval)

### 6-2. 질의 묶음별 세 모델 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">한 번 구한 정답 순위 배열을 묶음별로 나눕니다. <code style="font-size:0.95em;color:#345473;">Recall@5</code>는 정답이 상위 5개에 들어온 비율이고, <code style="font-size:0.95em;color:#345473;">Recall@1</code>·MRR은 상위 순위의 변화도 보여 줍니다.</p></div>

In [ ]:
# 같은 순위를 전체/학습에 나온 조문/처음 보는 조문으로 나눠 재사용합니다
query_groups = {
    "전체": np.full(len(labor_eval), True),
    "학습에 나온 조문 질의": ~is_unseen,
    "처음 보는 조문 질의": is_unseen,
}
model_ranks = {"베이스 모델": base_ranks, "쌍 학습 모델": tuned_ranks, "hard negative 학습 모델": hard_ranks}
compare_rows = []
for group, mask in query_groups.items():
    for name, ranks in model_ranks.items():
        scores = retrieval_scores(ranks[mask])  # mask가 True인 질의의 순위만 평가합니다.
        compare_rows.append({
            "질의 묶음": group,
            "모델": name,
            "질의 수": int(mask.sum()),
            "Recall@1": scores["Recall@1"],
            "Recall@5": scores["Recall@5"],
            "MRR": scores["MRR"],
        })
compare_table = pd.DataFrame(compare_rows)
display(compare_table)

### 6-3. 일반 분야 검색의 퇴행 점검

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01과 같은 <strong style="font-size:1.13em;color:#1f3a5f;">관광 43개·상식 30개·가상 도서관 10개 질문</strong>으로 세 모델을 비교합니다. 분야마다 자기 검색 문서 모음에서 평가하므로, 관광 388문서·43질의 실험도 그대로 유지됩니다.</p></div>

상식 30개는 Day 70의 질문·정답 쌍에서 질문을 검색어, 정답 문장을 검색 문서로 사용합니다. 예: “대한민국의 수도는 어디인가요?”로 “대한민국의 수도는 서울입니다.” 문서(`common:01`)를 찾습니다. 도서관 10개는 수업용 가상 규정입니다. 83개 검색 질의의 결과만으로 일반능력 전체의 변화를 단정하지 않습니다.

In [ ]:
# 학습에 쓰지 않은 동일한 일반 평가 자료를 읽습니다.
general_corpus = load_jsonl(data_dir / "본실습" / "general_corpus.jsonl")
general_eval = load_jsonl(data_dir / "본실습" / "general_eval.jsonl")
# 평가 쌍에는 분야 열이 없으므로 정답 doc_id를 문서의 분야와 연결합니다.
general_doc_domain = {doc["doc_id"]: doc["domain"] for doc in general_corpus}
general_domains = np.array([general_doc_domain[row["doc_id"]] for row in general_eval])
general_domain_names = ["관광", "상식", "도서관"]
general_doc_counts = {domain: sum(doc["domain"] == domain for doc in general_corpus) for domain in general_domain_names}

# 분야별 검색 결과를 계산해 모델별 순위 배열에 기록합니다.
general_model_ranks = {}
for name, model in [("베이스 모델", base_model), ("쌍 학습 모델", tuned_model), ("hard negative 학습 모델", hard_model)]:
    ranks = np.empty(len(general_eval), dtype=int)
    for domain in general_domain_names:
        domain_mask = general_domains == domain  # 현재 분야의 질의 위치만 True입니다.
        domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
        domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
        ranks[domain_mask] = gold_ranks(model, domain_corpus, domain_eval)
    general_model_ranks[name] = ranks

In [ ]:
# 모델별 순위에서 비교 기준인 베이스 순위를 꺼냅니다.
base_general_ranks = general_model_ranks["베이스 모델"]

# 전체·분야별 평가 범위를 준비합니다.
general_groups = {"전체(일반 분야)": np.full(len(general_eval), True)}  # True 83개로 전체 질의를 선택합니다.
general_groups.update({domain: general_domains == domain for domain in general_domain_names})  # 같은 분야 위치만 True입니다.

# 각 평가 범위에서 모델별 점수와 베이스 대비 변화량을 계산합니다.
general_rows = []
for group, mask in general_groups.items():
    before_r5 = float(np.mean(base_general_ranks[mask] <= 5))  # 이 묶음의 베이스 비율이며 반올림 전 값입니다.
    for name, ranks in general_model_ranks.items():
        scores = retrieval_scores(ranks[mask])  # mask가 True인 질의의 순위만 평가합니다.
        general_rows.append({
            "평가 묶음": group, "모델": name,
            "문서 수": len(general_corpus) if group == "전체(일반 분야)" else general_doc_counts[group],
            "질의 수": int(mask.sum()),  # True인 위치의 개수입니다.
            **scores,  # Recall@1·Recall@5·MRR 세 열을 합칩니다.
            "Recall@5 변화(pp)": round(100 * (float(np.mean(ranks[mask] <= 5)) - before_r5), 2),  # 비율 차이에 100을 곱합니다.
        })

# 점수와 변화량을 한 테이블로 표시합니다.
general_score_table = pd.DataFrame(general_rows)
display(general_score_table)

In [ ]:
# 노무 성능의 변화와 일반 검색의 변화폭을 함께 확인합니다.
change_rows = []
for group, baseline, models in [
    ("노무", base_ranks, {"쌍 학습 모델": tuned_ranks, "hard negative 학습 모델": hard_ranks}),
    ("일반 83질의", base_general_ranks, {name: ranks for name, ranks in general_model_ranks.items() if name != "베이스 모델"}),
]:
    before_r5 = float(np.mean(baseline <= 5))  # 반올림 전 비율로 변화량을 계산합니다.
    for name, ranks in models.items():
        after_r5 = float(np.mean(ranks <= 5))
        change_rows.append({"평가 분야": group, "모델": name, "베이스 Recall@5": round(before_r5, 3), "학습 후 Recall@5": round(after_r5, 3), "변화(pp)": round(100 * (after_r5 - before_r5), 2)})
regression_change_table = pd.DataFrame(change_rows)
display(regression_change_table)

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li><code style="font-size:0.95em;color:#345473;">변화(pp)</code>는 베이스 대비 퍼센트포인트 차이입니다. 음수이면 상위 5개 정답 질문이 줄고, 양수이면 늘며, 0이면 같습니다.</li>
<li>관광은 43질의라 한 건이 약 2.33pp, 도서관은 10질의라 한 건이 10pp를 바꿉니다. 질의 수가 적은 분야의 점수는 몇 건의 순위 변화에 크게 좌우됩니다.</li>
</ul></div>

## 7. 🖐️ 함께 따라하기: 증상으로 질환 문서 찾기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">본실습의 마이닝, 거짓 오답 빼기, triplet 학습을 의학 데이터에 적용해, 증상이 비슷한 다른 질환 문서를 hard negative로 넣었을 때 세 모델의 검색 점수를 직접 작성해 비교합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul>
<li><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> 교안 01 함께 따라하기와 같은 데이터입니다(검색 실습용이며 의학적 진단이나 조언이 아님).</li>
<li><strong style="font-size:1.13em;color:#1f3a5f;">문서 단위:</strong> 질환 문서 하나가 문서 하나입니다. 조·항이 없으므로 hard negative는 증상이 비슷한 다른 질환 문서입니다.</li>
<li><strong style="font-size:1.13em;color:#1f3a5f;">쌍 학습 모델:</strong> 마이닝에 쓸 쌍 학습 모델은 교안 01 함께 따라하기에서 저장한 <code style="font-size:0.95em;color:#345473;">output/med_tuned</code>를 불러옵니다. 폴더가 없으면 교안 01 함께 따라하기를 먼저 실행합니다.</li>
</ul></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 작성 셀의 <code style="font-size:0.95em;color:#345473;">...</code>만 주석에 맞춰 채웁니다. 완성 코드는 <code style="font-size:0.95em;color:#345473;">정답/교안_02_정답.ipynb</code>의 같은 절에 있습니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정</strong></p>
<ul>
<li><strong style="font-size:1.13em;color:#1f3a5f;">학습:</strong> 최대 5에폭, 배치 16, 학습률 5e-5입니다. 매 에폭 검증·저장, 검증 Recall@5 최적 복원, 2회 연속 미개선 종료는 쌍 학습과 같습니다.</li>
<li><strong style="font-size:1.13em;color:#1f3a5f;">마이닝:</strong> 본실습과 같이 <code style="font-size:0.95em;color:#345473;">relative_margin=0.05</code>, 질의마다 1개, <code style="font-size:0.95em;color:#345473;">&quot;top&quot;</code>입니다.</li>
</ul></div>

### 7-1. 데이터와 쌍 학습 모델

In [ ]:
# 의학 데이터: 질환 문서, 학습 쌍, 평가 질의
med_corpus = load_jsonl(data_dir / "따라하기" / "med_corpus.jsonl")
med_train = load_jsonl(data_dir / "따라하기" / "med_train.jsonl")
med_eval = load_jsonl(data_dir / "따라하기" / "med_eval.jsonl")
print("질환 문서:", len(med_corpus), "| 학습 쌍:", len(med_train), "| 평가 질의:", len(med_eval))

In [ ]:
# 문서는 doc_id, 질환 이름(disease), 본문(text) 세 칸입니다
med_corpus_table = pd.DataFrame(med_corpus)
display(med_corpus_table.head())

In [ ]:
# 프리픽스를 붙인 (질의, 정답 문서) 쌍 데이터셋(교안 01 함께 따라하기와 같은 코드)
med_pair_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in med_train],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in med_train],
})

# 마이닝에 쓸 쌍 학습 모델: 교안 01 함께 따라하기에서 저장한 폴더에서 불러옵니다
med_tuned_model = SentenceTransformer(str(output_dir / "med_tuned"), device=device)
display(med_pair_ds)

### 7-2. hard negative 마이닝

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">본실습의 <code style="font-size:0.95em;color:#345473;">mine_hard_negatives</code>처럼 쌍 학습 모델로 hard negative를 뽑아 triplet 데이터셋을 만듭니다.</p></div>

In [ ]:
from sentence_transformers.util import mine_hard_negatives

# 오답 문서 후보: 질환 문서 전체에 문서 프리픽스를 붙입니다
med_corpus_texts = [PASSAGE_PREFIX + doc["text"] for doc in med_corpus]

# 🖐️ 함께 따라하기: med_tuned_model 로 mine_hard_negatives 를 호출해 med_triplet_ds 를 만드세요
#                     (데이터셋 med_pair_ds, corpus med_corpus_texts, relative_margin 0.05, 질의마다 1개, "top")
med_triplet_ds = ...
print("오답 문서를 받은 행:", len(med_triplet_ds), "/", len(med_pair_ds))

### 7-3. 거짓 오답 빼기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">오답으로 학습시키기 조심스러운 문서 짝을 검토 목록으로 제외합니다. 이 데이터에는 조문 인용 규칙을 적용하지 않습니다.</p></div>

- **검토 목록:** 식중독/장염, 쓸개염/담석은 배포 문서에서 설명하는 증상이 겹칩니다. 질의에 둘 다 관련될 가능성이 있어 이 짝은 오답 학습에서 보수적으로 제외합니다. 질환 이름의 짝만으로 모든 질의의 정답 여부를 판정할 수는 없습니다.

In [ ]:
# 문서를 읽고 확인한 질환 짝(어느 쪽이 정답 칸에 와도 찾도록 두 순서를 모두 적습니다)
med_same_answer_pairs = [
    ("식중독", "장염"),
    ("장염", "식중독"),
    ("쓸개염", "담석"),
    ("담석", "쓸개염"),
]

# med_triplet_ds 의 문장에는 프리픽스가 붙어 있으므로, 프리픽스를 붙인 문서 text 로 문서를 찾습니다
med_passage_to_doc = {PASSAGE_PREFIX + doc["text"]: doc for doc in med_corpus}

med_kept_rows = []
med_removed_rows = []
for row in med_triplet_ds:
    pos_doc = med_passage_to_doc[row["positive"]]
    neg_doc = med_passage_to_doc[row["negative"]]

    # 🖐️ 함께 따라하기: (정답 질환, 오답 질환) 이름 짝이 med_same_answer_pairs 에 있으면 True 가 되게 채우세요
    in_list = ...

    # 검토 목록에 있는 질환 쌍은 오답 학습에서 제외합니다
    if in_list:
        med_removed_rows.append({"질의": row["anchor"][len(QUERY_PREFIX):], "정답 질환": pos_doc["disease"], "오답 질환": neg_doc["disease"]})
    else:
        med_kept_rows.append(row)

# 🖐️ 함께 따라하기: 남은 행 med_kept_rows 로 med_triplet_ds 를 다시 만드세요
med_triplet_ds = ...
print("뺀 행:", len(med_removed_rows), "| 학습에 쓸 행:", len(med_triplet_ds))

In [ ]:
# 제외한 행: 검토 목록과 겹쳐 오답 학습에서 보류한 행입니다
med_removed_table = pd.DataFrame(med_removed_rows)
display(med_removed_table)

### 7-4. Triplet으로 학습

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">본실습처럼 베이스 모델에서 새로 시작하고, 학습 데이터만 <code style="font-size:0.95em;color:#345473;">med_triplet_ds</code>로 바꿉니다. 검토에서 남은 행 수가 쌍 학습과 달라 갱신 횟수도 같다고 가정하지 않습니다. 의학 검증셋은 최적 에폭 선택용이므로 독립 테스트 점수는 아닙니다.</p></div>

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

# 베이스 모델 이름에서 새로 시작합니다
med_hard_model = SentenceTransformer(MODEL_ID, device=device)

# 의학 검증 손실과 전체 질환 문서에서의 검색 점수를 함께 기록합니다.
med_eval_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in med_eval],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in med_eval],
})
# 검증 질의마다 전체 문서에서 정답을 검색해 Recall@5를 계산합니다.
med_evaluator = InformationRetrievalEvaluator(
    # 질의 ID는 정답 매핑과 연결할 문자열 키이고, 값은 프리픽스가 붙은 질문입니다.
    queries={str(i): QUERY_PREFIX + row["anchor"] for i, row in enumerate(med_eval)},
    # 문서 ID -> 프리픽스가 붙은 검색 문서입니다.
    corpus={doc["doc_id"]: PASSAGE_PREFIX + doc["text"] for doc in med_corpus},
    # 질의 ID -> 정답 조문 ID 집합입니다.
    relevant_docs={str(i): {row["doc_id"]} for i, row in enumerate(med_eval)},
    name="med",  # 지표 이름에 포함되어 eval_med_cosine_recall@5를 만듭니다.
    score_functions={"cosine": cos_sim},  # 문서 점수는 코사인 유사도로 비교합니다.
    precision_recall_at_k=[1, 5],  # Recall@1과 Recall@5를 계산합니다.
    batch_size=64,  # 평가 문장을 64개씩 인코딩합니다.
    write_csv=False,
)

# 🖐️ 함께 따라하기: med_hard_model 로 MNR 손실을 만드세요
med_loss = ...

In [ ]:
# 최대 에폭·배치·학습률은 쌍 학습과 맞추지만 실제 갱신 횟수는 달라질 수 있습니다.
med_args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir / "med_hard_run"),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_steps=0.1,                           # 1보다 작은 실수는 비율. 전체 갱신 횟수의 처음 10% 동안 학습률을 0에서 올립니다
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # 질문·정답·오답 문자열을 모두 검사해 배치 내 중복을 피합니다.
    eval_strategy="epoch",                    # 매 에폭 검증 손실과 검색 점수를 계산합니다.
    save_strategy="epoch",
    load_best_model_at_end=True,               # 종료 후 검증 Recall@5 최적 모델을 복원합니다.
    # evaluator의 name=med, cosine 점수, Recall@5로 만들어진 지표 키입니다.
    metric_for_best_model="eval_med_cosine_recall@5",
    greater_is_better=True,                    # Recall@5는 값이 클수록 좋은 모델입니다.
    save_total_limit=2,
    logging_steps=5,
    report_to="none",
    seed=42,                                    # 배치를 묶는 순서를 고정해 다시 실행해도 같은 순서로 학습합니다
    use_cpu=(device.type == "cpu"),             # GPU(cuda)이면 False, CPU이면 True입니다.
)

In [ ]:
# 🖐️ 함께 따라하기: 학습/검증 데이터·evaluator·MNR 손실·조기 종료 콜백을 학습기에 연결해 학습하세요.
med_trainer = ...
med_hard_output = ...
display(med_hard_output)

# 매 에폭 검증 기록에서 손실과 검색 점수를 비교합니다.
med_epoch_history = pd.DataFrame([
    {"에폭": log["epoch"], "검증 MNR 손실": round(log["eval_loss"], 4),
     "검증 Recall@5": round(log["eval_med_cosine_recall@5"], 3)}
    for log in med_trainer.state.log_history if "eval_med_cosine_recall@5" in log
])
display(med_epoch_history)

# 복원된 최적 체크포인트와 선택 점수를 확인합니다.
med_best_checkpoint_table = pd.DataFrame([{
    "선택 기준": "검증 Recall@5 최대", "최적 Recall@5": round(med_trainer.state.best_metric, 3),
    "복원 체크포인트": med_trainer.state.best_model_checkpoint,
}])
display(med_best_checkpoint_table)

### 7-5. 세 모델 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">세 모델의 Recall@1·Recall@5·MRR을 테이블로 비교하되, 평가 질의 수가 적으므로 질의 한두 개 차이로 결론을 내리지 않습니다.</p></div>

In [ ]:
# 🖐️ 함께 따라하기: gold_ranks 로 세 모델(base_model, med_tuned_model, med_hard_model)의 의학 평가 질의별 정답 순위를 구하세요
med_base_ranks = ...
med_tuned_ranks = ...
med_hard_ranks = ...

med_table = pd.DataFrame(
    [retrieval_scores(med_base_ranks), retrieval_scores(med_tuned_ranks), retrieval_scores(med_hard_ranks)],
    index=["베이스 모델", "쌍 학습 모델", "hard negative 학습 모델"],
)
display(med_table)

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>이 표의 점수는 최적 에폭을 고를 때 사용한 검증셋에서 계산했습니다. 쌍 학습 모델과 Hard Negative 모델의 차이는 이 검증셋에서의 차이이며, 독립 테스트 성능은 아닙니다.</li>
</ul></div>

### 7-6. 다른 분야 검색의 변화

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">의학 검증</strong>은 가중치 학습에 넣지 않은 의학 질문으로 같은 분야의 일반화를 점검하고 최적 에폭을 선택합니다. <strong style="font-size:1.13em;color:#1f3a5f;">다른 분야 퇴행 점검</strong>은 의학 학습 전·후에 관광·상식·도서관 검색 성능이 유지되는지 확인하는 별도 평가입니다.</p></div>

앞의 일반 83질의와 베이스 순위를 재사용해 의학 쌍 학습·Hard Negative 학습 모델을 비교합니다. 분야마다 자기 후보 문서에서 검색하고, 계산한 순위를 원래 질의 순서에 넣습니다.

In [ ]:
# 베이스 순위는 재사용하고, 두 의학 모델도 같은 분야별 후보 문서에서 검색합니다.
med_general_model_ranks = {"베이스 모델": base_general_ranks}
for name, model in [("의학 쌍 학습 모델", med_tuned_model), ("의학 hard negative 학습 모델", med_hard_model)]:
    ranks = np.empty(len(general_eval), dtype=int)
    for domain in general_domain_names:
        domain_mask = general_domains == domain  # 현재 분야의 질의 위치만 True입니다.
        domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
        domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
        # 🖐️ 함께 따라하기: 현재 모델로 분야 문서·질의의 정답 순위를 구하세요.
        ranks[domain_mask] = ...
    med_general_model_ranks[name] = ranks  # 모델 이름에 83질의 순위를 연결합니다.

In [ ]:
# 전체 83질의와 각 분야의 점수·베이스 대비 변화를 함께 봅니다.
# 앞에서 만든 전체·분야별 범위와 모델별 순위로 점수·변화량을 계산합니다.
med_general_rows = []
for group, mask in general_groups.items():
    before_r5 = float(np.mean(base_general_ranks[mask] <= 5))  # 이 묶음의 베이스 비율이며 반올림 전 값입니다.
    for name, ranks in med_general_model_ranks.items():
        scores = retrieval_scores(ranks[mask])  # mask가 True인 질의의 순위만 평가합니다.
        med_general_rows.append({
            "평가 묶음": group, "모델": name,
            "문서 수": len(general_corpus) if group == "전체(일반 분야)" else general_doc_counts[group],
            "질의 수": int(mask.sum()),  # True인 위치의 개수입니다.
            **scores,  # Recall@1·Recall@5·MRR 세 열을 합칩니다.
            "Recall@5 변화(pp)": round(100 * (float(np.mean(ranks[mask] <= 5)) - before_r5), 2),  # 비율 차이에 100을 곱합니다.
        })

# 점수와 변화량을 한 테이블로 표시합니다.
med_general_score_table = pd.DataFrame(med_general_rows)
display(med_general_score_table)

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li><code style="font-size:0.95em;color:#345473;">Recall@5 변화(pp)</code>는 베이스 대비 퍼센트포인트 차이입니다. 음수이면 상위 5개 정답 질문이 줄고, 양수이면 늘며, 0이면 같습니다. 쌍 학습·Hard Negative 모델의 분야별 값을 비교합니다.</li>
<li>관광은 43질의, 도서관은 10질의이므로 한두 건의 순위 변화가 분야 점수에 크게 반영됩니다.</li>
</ul></div>

## 8. 핵심 정리

- **MNR 계산:** 질의와 문서의 유사도에 scale을 곱하고, 자기 정답을 고를 확률의 크로스 엔트로피를 줄입니다.
- **Hard Negative:** 모델이 헷갈리는 오답을 수집해 구분을 연습합니다. 정답 ID가 다르다는 사실만으로 실제 오답인지는 알 수 없습니다.
- **거짓 오답 검토:** 점수 간격과 내용 검토는 서로 보완합니다. 검토 목록·인용 규칙은 보수적 제외 기준이며 완전한 정답 판정이 아닙니다.
- **Triplet과 손실:** 데이터는 세 칸이어도 손실은 MNR입니다. 명시적 오답 문서는 배치의 다른 질의에도 후보가 됩니다.
- **모델 선택:** 최대 5에폭 중 검증 Recall@5가 가장 높은 체크포인트를 복원하고, 2회 연속 미개선이면 종료합니다. 검증 손실도 보되 일반 83질의는 선택에 사용하지 않습니다.
- **평가 해석:** 모델 선택용 검증 점수와 다른 분야 검색의 변화를 구분합니다. Hard Negative를 더했다고 모든 묶음이 좋아진다고 가정하지 않습니다.

## 9. 선택 실험: 마이닝 모델에 따른 후보 수

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">핵심 실습과 함께 따라하기를 마친 뒤 필요할 때만 실행합니다. 같은 규칙을 베이스 모델에도 적용해 마이닝 결과가 얼마나 달라지는지 비교합니다. 이 결과는 모델의 <strong style="font-size:1.13em;color:#1f3a5f;">검색 성능 순위가 아니라, 주어진 규칙으로 오답을 받은 학습 행 수</strong>입니다.</p></div>

쌍 학습 모델이 자기 학습 질의의 정답에 높은 점수를 주면 점수 간격 규칙을 쉽게 통과할 수 있습니다. 반대로 정답과 후보의 점수가 좁게 모이는 모델에서는 남는 행이 적을 수 있습니다. 어느 쪽도 행 수만으로 거짓 오답의 유무를 판정할 수 없습니다.

In [ ]:
# 본실습과 같은 후보·간격·개수 규칙을 베이스 모델에 적용합니다
base_mined = mine_hard_negatives(
    pair_ds,
    base_model,
    corpus=corpus_texts,
    relative_margin=0.05,
    num_negatives=1,
    sampling_strategy="top",
    batch_size=64,
    verbose=True,  # 후보를 못 찾은 행 안내는 규칙을 통과한 후보가 부족하다는 뜻입니다
)

# 쌍 학습 모델의 행 수는 내용 검토로 제외하기 전의 마이닝 결과입니다
mining_compare_table = pd.DataFrame({
    "마이닝 모델": ["베이스 모델", "쌍 학습 모델"],
    "학습 질의 수": [len(pair_ds), len(pair_ds)],
    "오답 후보를 받은 행": [len(base_mined), tuned_mined_count],
})
display(mining_compare_table)